In [5]:
# import sys
# !{sys.executable} -m pip install flask python-dotenv requests pillow azure-ai-textanalytics azure-cognitiveservices-vision-computervision msrest

In [13]:
# import sys
# !{sys.executable} -m pip install azure-cognitiveservices-speech

In [1]:
import flask
import requests
import PIL
import azure.ai.textanalytics
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
import azure.cognitiveservices.speech as speechsdk
from PIL import Image
import os
from dotenv import load_dotenv
from msrest.authentication import CognitiveServicesCredentials
import app

print("All libraries installed successfully!")

All libraries installed successfully!


### Initial Testing (before streamlining with app.py)

In [2]:
# Load environment variables from .env
load_dotenv()

# Read Azure Computer Vision credentials
VISION_KEY = os.getenv("VISION_KEY")
VISION_ENDPOINT = os.getenv("VISION_ENDPOINT")

if not VISION_KEY or not VISION_ENDPOINT:
    raise ValueError("Missing VISION_KEY or VISION_ENDPOINT in .env file.")

# Create Computer Vision client
vision_client = ComputerVisionClient(
    VISION_ENDPOINT,
    CognitiveServicesCredentials(VISION_KEY)
)

In [3]:
# Use local image
image_path = "images/prescription_0.png"  # put your image in project folder

with open(image_path, "rb") as image_stream:
    result = vision_client.read_in_stream(image_stream, raw=True)

# Get operation ID
operation_location = result.headers["Operation-Location"]
operation_id = operation_location.split("/")[-1]

# Wait for result
import time
while True:
    read_result = vision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted', 'running']:
        break
    time.sleep(1)

# Print extracted text
if read_result.status == 'succeeded':
    for page in read_result.analyze_result.read_results:
        for line in page.lines:
            print(line.text)

Dr. S Patel
MBBS MD
Rx
Tab Omeprazole 500mg 1-0-0
Tab Paracetamol 10mg 0-1-0
Tab Pantoprazole 200mg 0-0-1


In [4]:
# Load .env
load_dotenv()

LANGUAGE_KEY = os.getenv("LANGUAGE_KEY")
LANGUAGE_ENDPOINT = os.getenv("LANGUAGE_ENDPOINT")

url = f"{LANGUAGE_ENDPOINT}/language/:analyze-text?api-version=2023-04-01"

headers = {
    "Ocp-Apim-Subscription-Key": LANGUAGE_KEY,
    "Content-Type": "application/json"
}

data = {
    "kind": "EntityRecognition",
    "analysisInput": {
        "documents": [
            {
                "id": "1",
                "language": "en",
                "text": "Take Paracetamol 500mg three times a day after food"
            }
        ]
    }
}

response = requests.post(url, headers=headers, json=data)

print(response.status_code)
print(response.json())

200
{'kind': 'EntityRecognitionResults', 'results': {'documents': [{'id': '1', 'entities': [{'text': 'Paracetamol', 'category': 'Product', 'offset': 5, 'length': 11, 'confidenceScore': 1.0}, {'text': '500mg', 'category': 'Quantity', 'subcategory': 'Dimension', 'offset': 17, 'length': 5, 'confidenceScore': 0.99}, {'text': 'three', 'category': 'Quantity', 'subcategory': 'Number', 'offset': 23, 'length': 5, 'confidenceScore': 0.8}, {'text': 'a day', 'category': 'DateTime', 'subcategory': 'Duration', 'offset': 35, 'length': 5, 'confidenceScore': 0.79}], 'warnings': []}], 'errors': [], 'modelVersion': '2025-11-01'}}


Testing my CLU project

In [5]:
load_dotenv()

LANGUAGE_KEY = os.getenv("LANGUAGE_KEY")

# 👉 Use YOUR prediction URL here
url = "https://chatbotapp.cognitiveservices.azure.com/language/:analyze-conversations?api-version=2024-11-15-preview"

headers = {
    "Ocp-Apim-Subscription-Key": LANGUAGE_KEY,
    "Content-Type": "application/json"
}

data = {
    "kind": "Conversation",
    "analysisInput": {
        "conversationItem": {
            "id": "1",
            "text": "Explain this prescription",   # 🔁 change this for testing
            "modality": "text",
            "language": "en",
            "participantId": "1"
        }
    },
    "parameters": {
        "projectName": "PrescriptionAssistantCLU",
        "deploymentName": "deployment1",
        "verbose": True,
        "stringIndexType": "TextElement_V8"
    }
}

response = requests.post(url, headers=headers, json=data)

print(response.status_code)
print(response.json())

200
{'kind': 'ConversationResult', 'result': {'query': 'Explain this prescription', 'prediction': {'topIntent': 'ExplainPrescription', 'projectKind': 'Conversation', 'intents': [{'category': 'ExplainPrescription', 'confidenceScore': 0.9989982}, {'category': 'AskMedicinePurpose', 'confidenceScore': 0.82268065}, {'category': 'GeneralHelp', 'confidenceScore': 0.71220005}, {'category': 'AskDosage', 'confidenceScore': 0.68771}, {'category': 'AskUsageClarification', 'confidenceScore': 0.58832806}, {'category': 'None', 'confidenceScore': 0}], 'entities': [{'category': 'PrescriptionText', 'text': 'Explain', 'offset': 0, 'length': 7, 'confidenceScore': 1}, {'category': 'PrescriptionText', 'text': 'prescription', 'offset': 13, 'length': 12, 'confidenceScore': 1}]}}}


### New code with app.py

In [8]:
from PIL import Image

# Use local image
image_path = "images/prescription_0.png"  # put your image in project folder

**OCR Function**

In [9]:
print(app.extract_text_from_image(image_path))

Dr. S Patel
MBBS MD
Rx
Tab Omeprazole 500mg 1-0-0
Tab Paracetamol 10mg 0-1-0
Tab Pantoprazole 200mg 0-0-1


**CLU Function**

In [10]:
result = app.analyze_user_query_with_clu("Explain this prescription")
print(result)

{'top_intent': 'ExplainPrescription', 'intents': [{'category': 'ExplainPrescription', 'confidenceScore': 0.9989982}, {'category': 'AskMedicinePurpose', 'confidenceScore': 0.82268065}, {'category': 'GeneralHelp', 'confidenceScore': 0.71220005}, {'category': 'AskDosage', 'confidenceScore': 0.68771}, {'category': 'AskUsageClarification', 'confidenceScore': 0.58832806}, {'category': 'None', 'confidenceScore': 0}], 'entities': [{'category': 'PrescriptionText', 'text': 'Explain', 'offset': 0, 'length': 7, 'confidenceScore': 1}, {'category': 'PrescriptionText', 'text': 'prescription', 'offset': 13, 'length': 12, 'confidenceScore': 1}]}


In [11]:
result = app.process_prescription_request(
    image_path="images/prescription_0.png",
    user_text="Explain this prescription"
)

print(result["response_message"])
print("\n--- OCR TEXT ---")
print(result["ocr_text"])
print("\n--- CLU RESULT ---")
print(result["clu_result"])

Here is a simplified explanation of the prescription:

Dr. S Patel
MBBS MD
Rx
Tab Omeprazole 500mg 1-0-0
Tab Paracetamol 10mg 0-1-0
Tab Pantoprazole 200mg 0-0-1

Disclaimer: This is a demo assistant only and is not a substitute for advice from a doctor or pharmacist.

--- OCR TEXT ---
Dr. S Patel
MBBS MD
Rx
Tab Omeprazole 500mg 1-0-0
Tab Paracetamol 10mg 0-1-0
Tab Pantoprazole 200mg 0-0-1

--- CLU RESULT ---
{'top_intent': 'ExplainPrescription', 'intents': [{'category': 'ExplainPrescription', 'confidenceScore': 0.9989982}, {'category': 'AskMedicinePurpose', 'confidenceScore': 0.82268065}, {'category': 'GeneralHelp', 'confidenceScore': 0.71220005}, {'category': 'AskDosage', 'confidenceScore': 0.68771}, {'category': 'AskUsageClarification', 'confidenceScore': 0.58832806}, {'category': 'None', 'confidenceScore': 0}], 'entities': [{'category': 'PrescriptionText', 'text': 'Explain', 'offset': 0, 'length': 7, 'confidenceScore': 1}, {'category': 'PrescriptionText', 'text': 'prescription', 'of

In [12]:
result = app.process_prescription_request(
    user_text="What does after food mean"
)

print(result["response_message"])
print(result["clu_result"])

“After food” usually means the medicine should be taken after eating a meal.

Disclaimer: This is a demo assistant only and is not a substitute for advice from a doctor or pharmacist.
{'top_intent': 'AskUsageClarification', 'intents': [{'category': 'AskUsageClarification', 'confidenceScore': 0.9993982}, {'category': 'AskMedicinePurpose', 'confidenceScore': 0.53994477}, {'category': 'GeneralHelp', 'confidenceScore': 0.49458346}, {'category': 'AskDosage', 'confidenceScore': 0.43953055}, {'category': 'ExplainPrescription', 'confidenceScore': 0.4252181}, {'category': 'None', 'confidenceScore': 0}], 'entities': [{'category': 'TimingInstruction', 'text': 'after food', 'offset': 10, 'length': 10, 'confidenceScore': 1}]}


In [13]:
result = app.process_prescription_request(
    image_path="images/prescription_0.png",
    user_text="What does after food mean"
)

print(result["response_message"])

“After food” usually means the medicine should be taken after eating a meal.

Disclaimer: This is a demo assistant only and is not a substitute for advice from a doctor or pharmacist.


### Testing Azure Speech

In [14]:
# Load .env
load_dotenv()

speech_key = os.getenv("SPEECH_KEY")
speech_region = os.getenv("SPEECH_REGION")

# Configure speech
speech_config = speechsdk.SpeechConfig(
    subscription=speech_key,
    region=speech_region
)

# Output to speaker (or default audio device)
audio_config = speechsdk.audio.AudioOutputConfig(use_default_speaker=True)

# Create synthesizer
speech_synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=speech_config,
    audio_config=audio_config
)

# Test message
text = "Azure Speech service is working correctly."

print("Testing speech synthesis...")

result = speech_synthesizer.speak_text_async(text).get()

if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print("✅ Speech test successful!")
else:
    print("❌ Speech test failed:", result.reason)

Testing speech synthesis...
✅ Speech test successful!


In [7]:
load_dotenv()

speech_key = os.getenv("SPEECH_KEY")
speech_region = os.getenv("SPEECH_REGION")

def transcribe_audio_file(audio_path: str) -> str:
    speech_config = speechsdk.SpeechConfig(
        subscription=speech_key,
        region=speech_region
    )
    speech_config.speech_recognition_language = "en-US"

    audio_config = speechsdk.audio.AudioConfig(filename=audio_path)

    recognizer = speechsdk.SpeechRecognizer(
        speech_config=speech_config,
        audio_config=audio_config
    )

    result = recognizer.recognize_once_async().get()

    if result.reason == speechsdk.ResultReason.RecognizedSpeech:
        return result.text
    elif result.reason == speechsdk.ResultReason.NoMatch:
        return "No speech could be recognized."
    elif result.reason == speechsdk.ResultReason.Canceled:
        cancellation = result.cancellation_details
        return f"Speech recognition canceled: {cancellation.reason}"
    
    return "Speech recognition failed."

In [8]:
text = transcribe_audio_file("uploads/voice_test_1.wav")
print(text)

Testing the Azure Speech Recognition Service, can you transcribe the following tag text? Does it make sense to you?


### Running the Flask App

In [9]:
import sys
!"{sys.executable}" app.py

^C
